In [1]:
import logging
import re
import joblib
from typing import List, Tuple
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field

# --- CONFIGURACIÓN DE LOGS (Observabilidad) ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("API_Arancelaria")

app = FastAPI(
    title="Servicio de Clasificación Arancelaria HTS",
    description="""
    Microservicio de ML para la categorización automática de mercancías.
    Arquitectura: TF-IDF + Random Forest (Entrenado con 10,000 registros).
    """,
    version="1.1.0"
)

# --- LÓGICA DE PROCESAMIENTO ---

def limpiar_texto(texto: str) -> str:
    """
    Normaliza la descripción de la mercancía para asegurar consistencia con el modelo.
    
    Proceso:
    1. Conversión a minúsculas.
    2. Limpieza de caracteres no alfabéticos (conserva ñ y acentos).
    3. Eliminación de espacios excedentes.

    Args:
        texto (str): Descripción original del producto.

    Returns:
        str: Texto normalizado para el motor de inferencia.
    """
    if not texto:
        return ""
    texto = texto.lower()
    # Regex para mantener caracteres alfabéticos latinos
    texto = re.sub(r'[^a-zñáéíóú\s]', '', texto)
    return texto.strip()

# --- MODELOS DE DATOS (Validación Pydantic) ---

class SolicitudClasificacion(BaseModel):
    """Esquema de entrada para peticiones de clasificación."""
    descripcion: str = Field(
        ..., 
        min_length=3, 
        max_length=500,
        description="Descripción técnica de la mercancía."
    )
    top_k: int = Field(
        default=3, 
        ge=1, 
        le=5,
        description="Cantidad de sugerencias deseadas (Top-K)."
    )

class DetallePrediccion(BaseModel):
    """Estructura de una fracción arancelaria predicha."""
    fraccion_hts: str = Field(..., description="Código del Arancel Armonizado.")
    confianza: float = Field(..., description="Probabilidad asignada por el modelo (0-1).")

class RespuestaClasificacion(BaseModel):
    """Esquema de salida con el ranking de resultados."""
    descripcion_procesada: str
    predicciones: List[DetallePrediccion]

# --- GESTIÓN DEL MODELO (Patrón Singleton) ---

class GestorModelo:
    """
    Controlador del ciclo de vida del modelo de Machine Learning.
    Asegura que el pipeline se cargue una sola vez en la memoria RAM.
    """
    def __init__(self, ruta_archivo: str):
        self.ruta_archivo = ruta_archivo
        self.pipeline = None

    def cargar_modelo(self):
        """Carga el pipeline serializado (.joblib) desde el disco."""
        try:
            self.pipeline = joblib.load(self.ruta_archivo)
            logger.info(f"ÉXITO: Modelo cargado desde {self.ruta_archivo}")
        except Exception as e:
            logger.error(f"ERROR CRÍTICO: No se pudo cargar el archivo del modelo: {str(e)}")
            raise RuntimeError("El sistema de inferencia no está inicializado.")

    def ejecutar_inferencia(self, texto_crudo: str, k: int) -> List[Tuple[str, float]]:
        """
        Transforma el texto y predice las probabilidades de las fracciones.
        """
        texto_limpio = limpiar_texto(texto_crudo)
        
        # Obtención de probabilidades de cada clase (Softmax)
        probabilidades = self.pipeline.predict_proba([texto_limpio])[0]
        clases = self.pipeline.classes_
        
        # Obtener los índices de las K probabilidades más altas
        indices_top = probabilidades.argsort()[-k:][::-1]
        
        return [(clases[i], probabilidades[i]) for i in indices_top]

# Inicialización del gestor global
gestor = GestorModelo("clasificador_arancelario_v1.joblib")

@app.on_event("startup")
async def evento_inicio():
    """Se ejecuta al arrancar el servidor Uvicorn."""
    gestor.cargar_modelo()

# --- ENDPOINTS (Puntos de Acceso) ---

@app.post(
    "/v1/clasificar", 
    response_model=RespuestaClasificacion,
    status_code=status.HTTP_200_OK,
    tags=["Inferencia"],
    summary="Clasifica un producto a partir de su descripción técnica"
)
async def clasificar_producto(solicitud: SolicitudClasificacion):
    """
    Endpoint principal. Analiza la descripción mediante TF-IDF y Random Forest 
    para sugerir las fracciones arancelarias más probables.
    """
    if not gestor.pipeline:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE, 
            detail="Modelo no disponible en este momento."
        )
    
    try:
        resultados = gestor.ejecutar_inferencia(solicitud.descripcion, solicitud.top_k)
        
        # Mapeo a los modelos de Pydantic para la respuesta JSON
        lista_predicciones = [
            DetallePrediccion(fraccion_hts=res[0], confianza=res[1]) 
            for res in resultados
        ]
        
        return RespuestaClasificacion(
            descripcion_procesada=solicitud.descripcion,
            predicciones=lista_predicciones
        )
    except Exception as e:
        logger.error(f"Error durante la inferencia: {str(e)}")
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, 
            detail="Error interno en el motor de predicción."
        )

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_25956\2957227057.py:110: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
